# 181. RLOO：为什么简单 REINFORCE 也能做好 LLM RLHF？

> **面试问题：Leave-One-Out baseline 怎样降低方差？response token loss、KL shaping、stale rollout 和零方差组怎样实现？**

## 先给结论

RLOO 对同一 prompt 采 G 个响应，用其他 G-1 个响应的平均 reward 作为当前样本 baseline；它不需要 value network，且 baseline 不包含自身 reward。最终用 response token log-prob 乘 sequence advantage，并结合 reference KL。简单不等于无状态：rollout policy、reward/verifier、mask 与 batch 分组必须严格版本化。

## 推荐回答主线

1. 按 prompt 分组计算 leave-one-out baseline，证明优势和为零并对 reward 平移不变。
2. 把 sequence advantage 广播到 response token，prompt/padding 不参与 policy gradient。
3. 加入 per-token KL shaping、importance ratio/clip 和 rollout age，处理 online policy 漂移。
4. 监控 reward 方差、有效组、KL、长度、成本与独立评测，不把 reward 上升当质量结论。

## 教学实现边界

代码使用合成 log-prob/reward 展示 estimator，不执行真实模型采样；PPO/RLOO 实现细节、KL estimator 与分布式聚合必须以固定训练 recipe 为准。

## 一手资料

- [Back to Basics: REINFORCE Style RLHF](https://arxiv.org/abs/2402.14740)
- [REINFORCE](https://link.springer.com/article/10.1007/BF00992696)
- [Training language models to follow instructions](https://arxiv.org/abs/2203.02155)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
import warnings  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import numpy as np  # 导入本单元需要的依赖。

# 屏蔽当前运行环境由 torch 间接触发的 pynvml 弃用告警，不隐藏算法告警。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。
import torch  # 导入本单元需要的依赖。

# P 个 prompt、每个 G 个响应；reward 同时覆盖全同组与有区分度组。
torch.manual_seed(181)  # 计算并保存当前步骤的中间状态。
P, G, T = 3, 4, 6  # 计算并保存当前步骤的中间状态。
rewards = torch.tensor([[1.0, 0.0, 0.5, 1.5], [0.2, 0.2, 0.2, 0.2], [-1.0, 0.0, 1.0, 2.0]])  # 计算并保存当前步骤的中间状态。
response_mask = torch.tensor([[[1, 1, 1, 0, 0, 0]] * G, [[1, 1, 1, 1, 0, 0]] * G, [[1, 1, 1, 1, 1, 0]] * G], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。

assert rewards.shape == (P, G)  # 用受控断言验证关键不变量。
assert response_mask.shape == (P, G, T)  # 用受控断言验证关键不变量。
assert G > 1  # 用受控断言验证关键不变量。


## 1. Leave-One-Out baseline：每个样本只看同 prompt 的其他响应

`b_i=(sum_j r_j-r_i)/(G-1)`，优势 `A_i=r_i-b_i`。它等价于 `G/(G-1)*(r_i-group_mean)`，因此每组优势和为零；全同 reward 组没有学习信号。


In [ ]:
def rloo_advantages(group_rewards):  # 定义本节可复用的核心函数。
    if not isinstance(group_rewards, torch.Tensor) or group_rewards.ndim != 2:  # 按当前条件选择后续控制路径。
        raise TypeError("group_rewards 必须是二维 Tensor")  # 遇到非法合同立即显式失败。
    if not group_rewards.is_floating_point() or group_rewards.shape[1] <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("RLOO 需要每组至少两个浮点 reward")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(group_rewards).all():  # 按当前条件选择后续控制路径。
        raise ValueError("reward 必须全部有限")  # 遇到非法合同立即显式失败。
    total = group_rewards.sum(1, keepdim=True)  # 计算并保存当前步骤的中间状态。
    baseline = (total - group_rewards) / (group_rewards.shape[1] - 1)  # 计算并保存当前步骤的中间状态。
    return group_rewards - baseline, baseline  # 返回当前分支计算出的结果。

# 每组优势和为零，全同 reward 组优势全零，非法分组或 NaN 必须拒绝。
advantages, baselines = rloo_advantages(rewards)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(advantages.sum(1), torch.zeros(P))  # 用受控断言验证关键不变量。
assert torch.allclose(advantages[1], torch.zeros(G))  # 用受控断言验证关键不变量。
assert baselines.shape == rewards.shape  # 用受控断言验证关键不变量。
for invalid_rewards in (torch.tensor([[1.0]]), torch.tensor([[1.0, float("nan")]])):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        rloo_advantages(invalid_rewards); assert False  # 执行当前语句以推进本节示例。
    except (TypeError, ValueError):  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 2. 平移不变与 prompt-local：不能用全 batch 均值替代

给某个 prompt 全部 reward 加常数不应改变优势；如果用全 batch baseline，高 reward prompt 会系统性压制低 reward prompt，混入题目难度。RLOO 只比较同 prompt 候选。


In [ ]:
# 每个 prompt 加不同常数，RLOO 优势保持；全局中心化结果会变化。
shift = torch.tensor([[10.0], [-3.0], [5.0]])  # 计算并保存当前步骤的中间状态。
shifted = rewards + shift  # 计算并保存当前步骤的中间状态。
shifted_adv, _ = rloo_advantages(shifted)  # 计算并保存当前步骤的中间状态。
global_centered = rewards - rewards.mean()  # 计算并保存当前步骤的中间状态。
shifted_global = shifted - shifted.mean()  # 计算并保存当前步骤的中间状态。
assert torch.allclose(advantages, shifted_adv, atol=1e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(global_centered, shifted_global)  # 用受控断言验证关键不变量。
assert torch.allclose(advantages, G / (G - 1) * (rewards - rewards.mean(1, keepdim=True)))  # 用受控断言验证关键不变量。


## 3. Token log-prob：sequence advantage 只广播到有效 response

rollout 已给定采样 token，因此训练取这些 token 的新/旧 log-prob。prompt token、padding、截断后的无效位置都 mask；每条响应是一个 sequence reward，但梯度作用到其所有有效 response token。


In [ ]:
# 构造新旧策略对已采样 token 的 log-prob，mask 外放极值测试隔离。
new_logp = torch.randn(P, G, T, requires_grad=True)  # 计算并保存当前步骤的中间状态。
old_logp = (new_logp.detach() + 0.05 * torch.randn(P, G, T))  # 计算并保存当前步骤的中间状态。
new_logp_safe = torch.where(response_mask.bool(), new_logp, torch.zeros_like(new_logp))  # 计算并保存当前步骤的中间状态。
old_logp_safe = torch.where(response_mask.bool(), old_logp, torch.zeros_like(old_logp))  # 计算并保存当前步骤的中间状态。
sequence_new = (new_logp_safe * response_mask).sum(-1)  # 计算并保存当前步骤的中间状态。
sequence_old = (old_logp_safe * response_mask).sum(-1)  # 计算并保存当前步骤的中间状态。
assert sequence_new.shape == (P, G)  # 用受控断言验证关键不变量。
assert torch.equal(new_logp_safe[response_mask == 0], torch.zeros_like(new_logp_safe[response_mask == 0]))  # 用受控断言验证关键不变量。
assert response_mask.sum(-1).min() > 0  # 用受控断言验证关键不变量。


## 4. RLOO policy loss：baseline 与 reward 必须 stop-gradient

最小形式是 `-A * sum_t logπ(y_t)`。若用 importance ratio，可在 sequence 或 token 层构造并裁剪；这里先展示 on-policy estimator，并确保 advantage 不反传进 reward 模型。


In [ ]:
def rloo_policy_loss(sequence_logp, advantage, raw_ratio=None, clipped_ratio=None):  # 定义本节可复用的核心函数。
    if sequence_logp.shape != advantage.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("sequence log-prob 与 advantage 形状必须一致")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(sequence_logp).all() or not torch.isfinite(advantage).all():  # 按当前条件选择后续控制路径。
        raise ValueError("policy loss 输入必须有限")  # 遇到非法合同立即显式失败。
    fixed_advantage = advantage.detach()  # 计算并保存当前步骤的中间状态。
    if raw_ratio is None and clipped_ratio is None:  # 按当前条件选择后续控制路径。
        return -(sequence_logp * fixed_advantage).mean()  # 返回当前分支计算出的结果。
    if raw_ratio is None or clipped_ratio is None:  # 按当前条件选择后续控制路径。
        raise ValueError("importance ratio 与 clipped ratio 必须成对提供")  # 遇到非法合同立即显式失败。
    if raw_ratio.shape != advantage.shape or clipped_ratio.shape != advantage.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("importance ratio 形状错误")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(raw_ratio).all() or not torch.isfinite(clipped_ratio).all():  # 按当前条件选择后续控制路径。
        raise ValueError("importance ratio 必须有限")  # 遇到非法合同立即显式失败。
    # PPO 风格 surrogate：ratio 承担 dπ/dlogπ，不能再额外乘一次 sequence_logp。
    return -torch.minimum(raw_ratio * fixed_advantage, clipped_ratio * fixed_advantage).mean()  # 返回当前分支计算出的结果。

# on-policy 梯度符号正确，mask 外 token 无梯度；reward/baseline 不参与反传。
policy_loss = rloo_policy_loss(sequence_new, advantages)  # 计算并保存当前步骤的中间状态。
policy_loss.backward(retain_graph=True)  # 计算并保存当前步骤的中间状态。
positive = advantages > 0  # 计算并保存当前步骤的中间状态。
assert torch.all(new_logp.grad.sum(-1)[positive] < 0)  # 用受控断言验证关键不变量。
assert new_logp.grad[response_mask == 0].abs().sum() == 0  # 用受控断言验证关键不变量。
assert advantages.requires_grad is False  # 用受控断言验证关键不变量。


## 5. Per-token KL shaping：成本应与实际 response 长度对齐

常把 `-beta*(logπ-logπ_ref)` 作为每 token non-score reward，再与末端任务 reward 合成 return。长响应会累计更多 KL，因此要监控长度；reference log-prob 固定。


In [ ]:
def shaped_sequence_reward(task_reward, policy_token_logp, reference_token_logp, mask, beta):  # 定义本节可复用的核心函数。
    if task_reward.ndim != 2 or policy_token_logp.shape != reference_token_logp.shape or policy_token_logp.shape != mask.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("reward、log-prob 或 mask 形状错误")  # 遇到非法合同立即显式失败。
    if tuple(policy_token_logp.shape[:2]) != tuple(task_reward.shape):  # 按当前条件选择后续控制路径。
        raise ValueError("token 张量的分组维必须与 reward 对齐")  # 遇到非法合同立即显式失败。
    if not isinstance(beta, (int, float)) or not math.isfinite(beta) or beta < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("KL beta 必须是有限非负数")  # 遇到非法合同立即显式失败。
    tensors = (task_reward, policy_token_logp, reference_token_logp, mask)  # 计算并保存当前步骤的中间状态。
    if not all(torch.isfinite(tensor).all() for tensor in tensors):  # 按当前条件选择后续控制路径。
        raise ValueError("KL shaping 输入必须全部有限")  # 遇到非法合同立即显式失败。
    if not torch.all((mask == 0) | (mask == 1)):  # 按当前条件选择后续控制路径。
        raise ValueError("response mask 必须是 0/1")  # 遇到非法合同立即显式失败。
    # rollout reward 使用冻结 behavior/reference log-prob；显式 detach，learner 更新不能改写 advantage。
    sampled_log_ratio = policy_token_logp.detach() - reference_token_logp.detach()  # 计算并保存当前步骤的中间状态。
    token_kl_penalty = -beta * sampled_log_ratio * mask  # 计算并保存当前步骤的中间状态。
    return task_reward.detach() + token_kl_penalty.sum(-1), token_kl_penalty  # 返回当前分支计算出的结果。

# shaped reward 由 behavior policy 固定，并立即进入 LOO advantage；beta=0 回到任务 reward。
reference_token_logp = old_logp_safe - 0.03  # 计算并保存当前步骤的中间状态。
shaped, kl_tokens = shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, 0.1)  # 计算并保存当前步骤的中间状态。
shaped_advantages, shaped_baselines = rloo_advantages(shaped)  # 计算并保存当前步骤的中间状态。
zero_beta, _ = shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, 0.0)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(zero_beta, rewards)  # 用受控断言验证关键不变量。
assert torch.allclose(shaped_advantages.sum(1), torch.zeros(P), atol=1e-6)  # 用受控断言验证关键不变量。
assert kl_tokens[response_mask == 0].abs().sum() == 0  # 用受控断言验证关键不变量。
assert torch.isfinite(shaped_advantages).all()  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    shaped_sequence_reward(rewards, old_logp_safe, reference_token_logp, response_mask, float("nan")); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 6. Stale rollout：importance ratio 与 policy version 必须显式处理

生成和训练解耦时，数据来自旧 policy。ratio 过大意味着 estimator 高方差或越过 trust region；可限制 rollout age、裁剪 ratio 或丢弃，且日志保存行为策略 log-prob。


In [ ]:
def clipped_importance_ratio(new_sequence_logp, behavior_sequence_logp, clip=0.2, max_abs_log_ratio=30.0):  # 定义本节可复用的核心函数。
    if new_sequence_logp.shape != behavior_sequence_logp.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("新旧 sequence log-prob 形状必须一致")  # 遇到非法合同立即显式失败。
    if not all(torch.isfinite(x).all() for x in (new_sequence_logp, behavior_sequence_logp)):  # 按当前条件选择后续控制路径。
        raise ValueError("新旧 sequence log-prob 必须有限")  # 遇到非法合同立即显式失败。
    if not all(isinstance(x, (int, float)) and math.isfinite(x) for x in (clip, max_abs_log_ratio)):  # 按当前条件选择后续控制路径。
        raise ValueError("ratio 配置必须有限")  # 遇到非法合同立即显式失败。
    if not 0 <= clip < 1 or max_abs_log_ratio <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("clip 或 log-ratio 门禁非法")  # 遇到非法合同立即显式失败。
    log_ratio = new_sequence_logp - behavior_sequence_logp  # 计算并保存当前步骤的中间状态。
    if log_ratio.abs().max() > max_abs_log_ratio:  # 按当前条件选择后续控制路径。
        raise ValueError("policy 漂移过大，拒绝可能溢出的 rollout")  # 遇到非法合同立即显式失败。
    raw = torch.exp(log_ratio)  # 计算并保存当前步骤的中间状态。
    return raw, raw.clamp(1 - clip, 1 + clip)  # 返回当前分支计算出的结果。

def rollout_acceptable(current_version, rollout_version, max_age):  # 定义本节可复用的核心函数。
    if any(type(value) is not int for value in (current_version, rollout_version, max_age)):  # 按当前条件选择后续控制路径。
        raise TypeError("policy version 与 max_age 必须是整数")  # 遇到非法合同立即显式失败。
    if min(current_version, rollout_version, max_age) < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("policy version 与 max_age 不得为负")  # 遇到非法合同立即显式失败。
    return 0 <= current_version - rollout_version <= max_age  # 返回当前分支计算出的结果。

def rloo_training_objective(task_reward, policy_token_logp, behavior_token_logp, reference_token_logp, mask, beta, clip, current_version, rollout_version, max_age):  # 定义本节可复用的核心函数。
    if not rollout_acceptable(current_version, rollout_version, max_age):  # 按当前条件选择后续控制路径。
        raise ValueError("rollout 已过期或来自未来 policy")  # 遇到非法合同立即显式失败。
    # reward/advantage 固定在采样时的 behavior policy；learner 只通过 ratio/surrogate 接收梯度。
    shaped_reward, kl_penalty = shaped_sequence_reward(task_reward, behavior_token_logp, reference_token_logp, mask, beta)  # 计算并保存当前步骤的中间状态。
    advantage, baseline = rloo_advantages(shaped_reward)  # 计算并保存当前步骤的中间状态。
    sequence_policy = (policy_token_logp * mask).sum(-1)  # 计算并保存当前步骤的中间状态。
    sequence_behavior = (behavior_token_logp.detach() * mask).sum(-1)  # 计算并保存当前步骤的中间状态。
    raw_ratio, clipped_ratio = clipped_importance_ratio(sequence_policy, sequence_behavior, clip)  # 计算并保存当前步骤的中间状态。
    loss = rloo_policy_loss(sequence_policy, advantage, raw_ratio, clipped_ratio)  # 计算并保存当前步骤的中间状态。
    return {"loss": loss, "reward": shaped_reward, "advantage": advantage, "baseline": baseline, "kl_tokens": kl_penalty, "raw_ratio": raw_ratio, "clipped_ratio": clipped_ratio}  # 返回当前分支计算出的结果。

# KL、importance ratio 与版本门禁进入同一目标；改变 learner 不得改写 rollout reward/advantage。
training = rloo_training_objective(rewards, new_logp_safe, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, 10, 8, 2)  # 计算并保存当前步骤的中间状态。
changed_learner = new_logp_safe + 0.01 * response_mask  # 计算并保存当前步骤的中间状态。
changed_training = rloo_training_objective(rewards, changed_learner, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, 10, 8, 2)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(training["loss"])  # 用受控断言验证关键不变量。
assert torch.allclose(training["advantage"], shaped_advantages)  # 用受控断言验证关键不变量。
assert torch.equal(training["reward"], changed_training["reward"])  # 用受控断言验证关键不变量。
assert torch.equal(training["advantage"], changed_training["advantage"])  # 用受控断言验证关键不变量。
assert not torch.allclose(training["raw_ratio"], changed_training["raw_ratio"])  # 用受控断言验证关键不变量。
assert ((training["clipped_ratio"] >= 0.8) & (training["clipped_ratio"] <= 1.2)).all()  # 用受控断言验证关键不变量。
objective_gradient = torch.autograd.grad(training["loss"], new_logp, retain_graph=True)[0]  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(objective_gradient).all() and objective_gradient.abs().sum() > 0  # 用受控断言验证关键不变量。
assert objective_gradient[response_mask == 0].abs().sum() == 0  # 用受控断言验证关键不变量。
for bad_versions in ((10, 7, 2), (10, 11, 2)):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        rloo_training_objective(rewards, new_logp_safe, old_logp_safe, reference_token_logp, response_mask, 0.1, 0.2, *bad_versions); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。
for bad_ratio_args in ((torch.tensor([[100.0]]), torch.zeros(1, 1), 0.2), (torch.zeros(1, 1), torch.zeros(1, 1), -0.1)):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        clipped_importance_ratio(*bad_ratio_args); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    rollout_acceptable(1, 0, -1); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 7. 有效学习组：全同 reward、解析失败与 reward 饱和要分开报告

组内零方差不会产生 RLOO 信号；可能因为任务太易/太难、verifier 崩溃或 reward 被截平。面板应报告有效组比例、组内标准差、成功率、parse/verifier error 与长度。


In [ ]:
def group_signal_report(group_rewards, eps=1e-8):  # 定义本节可复用的核心函数。
    if not isinstance(eps, (int, float)) or not math.isfinite(eps) or eps < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("eps 必须有限非负")  # 遇到非法合同立即显式失败。
    rloo_advantages(group_rewards)  # 执行当前语句以推进本节示例。
    std = group_rewards.std(1, unbiased=False)  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "informative_fraction": float((std > eps).float().mean()),  # 执行当前语句以推进本节示例。
        "mean_group_std": float(std.mean()),  # 执行当前语句以推进本节示例。
        "zero_groups": int((std <= eps).sum()),  # 计算并保存当前步骤的中间状态。
    }  # 执行当前语句以推进本节示例。

# 面板读取真正用于训练的 shaped rollout reward，而不是脱离主路径的 raw task reward。
report = group_signal_report(training["reward"].detach())  # 计算并保存当前步骤的中间状态。
assert report["zero_groups"] == 1  # 用受控断言验证关键不变量。
assert math.isclose(report["informative_fraction"], 2 / 3, abs_tol=1e-6)  # 用受控断言验证关键不变量。
assert report["mean_group_std"] >= 0  # 用受控断言验证关键不变量。


## 8. 制品与发布：rollout、reward、reference 和 learner 是四个版本

保存 prompt id、response tokens/mask、behavior log-prob、policy/ref/reward/verifier hash、采样参数和时间。发布用独立 evaluator、人评、安全 slice、KL、长度、pass@1 与每有效组成本验收。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class RLOOArtifact:  # 定义承载本节状态与行为的数据结构。
    learner_policy: str  # 执行当前语句以推进本节示例。
    behavior_policy: str  # 执行当前语句以推进本节示例。
    reference: str  # 执行当前语句以推进本节示例。
    reward: str  # 执行当前语句以推进本节示例。
    verifier: str  # 执行当前语句以推进本节示例。
    group_size: int  # 执行当前语句以推进本节示例。
    kl_beta: float  # 执行当前语句以推进本节示例。
    ratio_clip: float  # 执行当前语句以推进本节示例。
    max_rollout_age: int  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# learner/behavior/reference 四类版本与目标配置进入摘要，改变 reward 产生不同制品。
artifact = RLOOArtifact("policy-v10", "policy-v8", "ref-v1", "rm-v5", "verify-v3", G, 0.1, 0.2, 2)  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert artifact.group_size == rewards.shape[1]  # 用受控断言验证关键不变量。
assert artifact.learner_policy != artifact.behavior_policy  # 用受控断言验证关键不变量。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != artifact_hash(RLOOArtifact("policy-v10", "policy-v8", "ref-v1", "rm-v6", "verify-v3", G, 0.1, 0.2, 2))  # 用受控断言验证关键不变量。


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
